In [0]:
from typing import Any

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, when

import logging


In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = "../../helpers/lng_lnd"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = "../../helpers/curated_helpers"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_portfolio(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "PortfolioId Int,"
        # "PortfolioName String,"
        # "vendor_code_qualco Int,"
        # "unit_code_profits Int,"
        # "portfolio_id_sme Int,"
        # "portfolio_id_thesis Int,"
        # "portfolio_id_qca Int,"
        # "portfolio_id_optiverse Decimal(8,2),"
        # "AssignmentDate Timestamp,"
        # "originBank String,"
        # "Prefix String,"
        # "TiresiasCode String,"
        # "TiresiasName String,"
        # "ParentPortfolio String,"
        # "PortfolioKey String,"
        # "spv_qualco_AB Decimal(8,2),"
        # "InvestorCountry String,"
        # "InvestorCountryCode String,"
        # "AssetOwner String,"
        # "profitCenter Int,"
        # "processSettlementFlag boolean,"
        # "IsSingleTicketFlag int,"
        # "IsBankingFlag boolean,"
        # "IsSecuritizedFlag boolean,"
        # "IsHAPSFlag boolean,"
        # "isREoCOFlag boolean,"
        # "isCostPlusFlag boolean,"
        # "GBVInitializationDate date,"
        # "RetailWholesale string,"
        # "closingdate date,"
        # "Type_of_Portfolio string,"
        # "Short_Name_Asset_Owner string,"
        # "Short_Name_Portfolio_Name string,"
        # "OwnerId int,"
        # "CommonPortfolioID string,"
        # "CommonPortfolio string,"        
        # "Datadate Timestamp,"
        # "BusinessDate Date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.portfolio"
        portfolio_input_path = (
            f"s3a://{env_prefix}-ec1-cepal-common-zone/portfolio/portfolio.json"
        )

        # ================== Create Delta Table ==================

        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")
            
            
        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")


        # ==================Load Portfolio json ==================

        portfolio_df = spark.read.json(portfolio_input_path)

        portfolio_df = portfolio_df.withColumn("BusinessDate", lit(run_date)
                                ).withColumn("CommonPortfolioID",
                                             when(portfolio_df.PortfolioId.isin(19,20,21), 18)
                                             .when(portfolio_df.PortfolioId.isin(34,35), 36)
                                             .when(portfolio_df.PortfolioId.isin(47,48, 50), 49)
                                             .when(portfolio_df.PortfolioId.isin(64,65), 66)
                                             .when(portfolio_df.PortfolioId.isin(69,70), 71)
                                             .when(portfolio_df.PortfolioId.isin(72,73), 74)
                                             .when(portfolio_df.PortfolioId.isin(75,76,77), 78)
                                ).withColumn("CommonPortfolio",
                                             when(portfolio_df.PortfolioId.isin(19,20,21), 'OrionGalaxy')
                                             .when(portfolio_df.PortfolioId.isin(34,35), 'CosmosRetailWhl')
                                             .when(portfolio_df.PortfolioId.isin(47,48, 50), 'HermesABC')
                                             .when(portfolio_df.PortfolioId.isin(64,65), 'MonzaRetailWhl')
                                             .when(portfolio_df.PortfolioId.isin(69,70), 'DomusRetailWhl')
                                             .when(portfolio_df.PortfolioId.isin(72,73), 'RhodiumRetailWhl')
                                             .when(portfolio_df.PortfolioId.isin(75,76,77), 'GaiaRetailWhl')
                                )

        portfolio_df = portfolio_df.select(
            portfolio_df.PortfolioId.cast("Int"),
            portfolio_df.PortfolioName.cast("String"),
            portfolio_df.vendor_code_qualco.cast("Int"),
            portfolio_df.unit_code_profits.cast("Int"),
            portfolio_df.portfolio_id_sme.cast("Int"),
            portfolio_df.portfolio_id_thesis.cast("Int"),
            portfolio_df.portfolio_id_qca.cast("Int"),
            portfolio_df.portfolio_id_optiverse.cast("Decimal(8,2)"),
            portfolio_df.AssignmentDate.cast("Timestamp"),
            portfolio_df.originBank.cast("String"),
            portfolio_df.Prefix.cast("String"),
            portfolio_df.TiresiasCode.cast("String"),
            portfolio_df.TiresiasName.cast("String"),
            portfolio_df.ParentPortfolio.cast("String"),
            portfolio_df.PortfolioKey.cast("String"),
            portfolio_df.spv_qualco_AB.cast("Decimal(8,2)"),
            portfolio_df.InvestorCountry.cast("String"),
            portfolio_df.InvestorCountryCode.cast("String"),
            portfolio_df.AssetOwner.cast("String"),
            portfolio_df.profitCenter.cast("Int"),
            portfolio_df.processSettlementFlag.cast("boolean"),
            portfolio_df.IsSingleTicketFlag.cast("Int"),
            portfolio_df.IsBankingFlag.cast("boolean"),
            portfolio_df.IsSecuritizedFlag.cast("boolean"),
            portfolio_df.IsHAPSFlag.cast("boolean"),
            portfolio_df.isREoCOFlag.cast("boolean"),
            portfolio_df.isCostPlusFlag.cast("boolean"),
            portfolio_df.GBVInitializationDate.cast("date"),
            portfolio_df.RetailWholesale.cast("string"),
            portfolio_df.closingdate.cast("date"),
            portfolio_df.Type_of_Portfolio.cast("string"),
            portfolio_df.Short_Name_Asset_Owner.cast("string"),
            portfolio_df.Short_Name_Portfolio_Name.cast("string"),
            portfolio_df.OwnerId.cast("int"),
            portfolio_df.CommonPortfolioID.cast("string"),
            portfolio_df.CommonPortfolio.cast("string"),
            portfolio_df.Datadate.cast("Timestamp"),
            portfolio_df.Arotron_dec_flag.cast("Int"),
            portfolio_df.AssetOwnerTaxID.cast("String"),
            portfolio_df.CollectionBankFile.cast("String"),
            portfolio_df.BusinessDate.cast("Date"),
        )

        # ================== Append ==================

        portfolio_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in locals().items()
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_portfolio(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)